# 04 Silver Layer – Healthcare (Simple + Validations)

This notebook keeps a short, direct teaching structure while preserving the Silver-layer rules and safeguards from the detailed healthcare implementation.

It reads the six Bronze tables, cleans and casts the data, checks healthcare rules, separates invalid rows into quarantine, removes duplicates with a deterministic audit, checks relationships and business consistency, and writes the Silver outputs.

**Schemas**
- Bronze: `capstone_8hospital.bronze_schema` – read only
- Silver: `capstone_8hospital.silver_schema` – output tables

Run the Bronze ingestion and profiling notebooks before running this notebook.\n\n**Parity fixes in this version**\n- Trims all string columns before validation while preserving `_raw_record` from the original Bronze row.\n- Uses the same deterministic deduplication ordering as the detailed Python implementation.\n- Checks relationship columns before joins and reports missing columns clearly.\n- Includes the DQ rule register (DQ001–DQ008).\n- Skips optional chronology comparisons when either timestamp column is absent.\n

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from functools import reduce

CATALOG = "capstone_8hospital"
BRONZE_SCHEMA = "bronze_schema"
SILVER_SCHEMA = "silver_schema"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

def bronze(table):
    return f"{CATALOG}.{BRONZE_SCHEMA}.{table}"

def silver(table):
    return f"{CATALOG}.{SILVER_SCHEMA}.{table}"

print("Bronze:", CATALOG + "." + BRONZE_SCHEMA)
print("Silver:", CATALOG + "." + SILVER_SCHEMA)

## 1. Read the six Bronze tables

In [0]:
encounters = spark.table(bronze("bronze_patient_encounters"))
labs = spark.table(bronze("bronze_lab_test_events"))
transfers = spark.table(bronze("bronze_bed_transfer_events"))
patients = spark.table(bronze("bronze_patients"))
departments = spark.table(bronze("bronze_departments"))
beds = spark.table(bronze("bronze_beds"))

source_tables = {
    "patient_encounters": encounters,
    "lab_test_events": labs,
    "bed_transfer_events": transfers,
    "patient_master": patients,
    "department_master": departments,
    "bed_master": beds
}

print("BRONZE TABLE COUNTS")
for name, df in source_tables.items():
    print(name, "=", df.count())

# Save the current Bronze Delta versions so we can confirm they were not changed.
def get_table_version(table_name):
    try:
        return spark.sql(f"DESCRIBE HISTORY {table_name} LIMIT 1").first()["version"]
    except Exception:
        return None

bronze_versions_before = {
    name: get_table_version(bronze(table))
    for name, table in {
        "encounters": "bronze_patient_encounters",
        "labs": "bronze_lab_test_events",
        "transfers": "bronze_bed_transfer_events",
        "patients": "bronze_patients",
        "departments": "bronze_departments",
        "beds": "bronze_beds"
    }.items()
}
print("Saved Bronze table versions for the final read-only check.")


## 2. Trim spaces from text columns\nThe function applies `trim()` to every string column in all six inputs, matching the detailed implementation. `_raw_record` is captured before this transformation so the original Bronze values remain auditable.\n

In [0]:
def trim_strings(df):
    for name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(name, F.trim(F.col(name)))
    return df

encounters = trim_strings(encounters)
labs = trim_strings(labs)
transfers = trim_strings(transfers)
patients = trim_strings(patients)
departments = trim_strings(departments)
beds = trim_strings(beds)

print("Completed trimming spaces from text columns.")

## 3. Prepare for data type conversion

The actual conversions happen in Section 5. Doing the conversion there allows the notebook to detect values that cannot be converted and send those records to quarantine instead of losing the original invalid value.

In [0]:
print("Data type conversion will be applied in Section 5, where failed conversions can also be detected.")

## 4. Define the healthcare data-quality rules

These rules are taken from the detailed healthcare notebook:

- Required fields must not be empty.
- Configured date, timestamp and numeric fields must convert successfully.
- Categorical columns must contain an allowed value.
- Floor must be zero or more; department bed capacity must be at least one.
- Event timestamps must follow the correct order.

Optional fields are not rejected just because they are NULL.

In [0]:
TYPE_MAP = {
    "patient_encounters": {"admission_ts": "timestamp", "discharge_ts": "timestamp"},
    "lab_test_events": {"ordered_ts": "timestamp", "collected_ts": "timestamp", "result_ts": "timestamp"},
    "bed_transfer_events": {"transfer_ts": "timestamp", "recorded_ts": "timestamp"},
    "patient_master": {"registration_date": "date"},
    "department_master": {"floor": "int", "capacity_beds": "int"},
    "bed_master": {"floor": "int", "effective_date": "date"}
}

BUSINESS_KEYS = {
    "patient_encounters": "encounter_id",
    "lab_test_events": "lab_event_id",
    "bed_transfer_events": "transfer_id",
    "patient_master": "patient_id",
    "department_master": "department_id",
    "bed_master": "bed_id"
}

REQUIRED_FIELDS = {
    "patient_encounters": ["encounter_id", "patient_id", "department_id", "admission_ts", "encounter_type", "triage_priority", "encounter_status"],
    "lab_test_events": ["lab_event_id", "encounter_id", "patient_id", "test_code", "ordered_ts", "result_status"],
    "bed_transfer_events": ["transfer_id", "encounter_id", "patient_id", "from_bed_id", "to_bed_id", "transfer_ts", "transfer_status"],
    "patient_master": ["patient_id"],
    "department_master": ["department_id"],
    "bed_master": ["bed_id"]
}

CATEGORY_RULES = {
    "patient_encounters": {
        "encounter_type": ["EMERGENCY", "INPATIENT", "OUTPATIENT"],
        "triage_priority": ["P1", "P2", "P3", "P4"],
        "encounter_status": ["DISCHARGED", "TRANSFERRED"],
        "admission_source": ["EMERGENCY", "REFERRAL", "WALK_IN"],
        "diagnosis_group": ["CARDIAC", "GENERAL", "RESPIRATORY", "TRAUMA"]
    },
    "lab_test_events": {
        "test_code": ["CBC", "GLUCOSE", "KFT", "LFT"],
        "result_status": ["COLLECTED", "ORDERED", "RESULTED"],
        "priority": ["ROUTINE", "STAT", "URGENT"]
    },
    "bed_transfer_events": {
        "transfer_reason": ["CAPACITY", "CLINICAL", "ISOLATION"],
        "transfer_status": ["ACTIVE", "INACTIVE", "PENDING"]
    },
    "patient_master": {
        "age_band": ["0-17", "18-35", "36-55", "56-75"],
        "sex": ["F", "M", "OTHER"],
        "consent_flag": ["N", "Y"],
        "insurance_type": ["BASIC", "PREMIUM", "STANDARD"],
        "patient_status": ["ACTIVE", "INACTIVE", "PENDING"],
        "primary_language": ["Bengali", "English", "Hindi", "Kannada", "Tamil", "Telugu"]
    },
    "department_master": {
        "service_level": ["EMERGENCY", "INPATIENT"],
        "active_flag": ["N", "Y"]
    },
    "bed_master": {
        "ward": ["GENERAL", "ICU", "SURGICAL"],
        "bed_type": ["ICU", "STANDARD"],
        "isolation_capable": ["N", "Y"],
        "bed_status": ["ACTIVE", "INACTIVE", "PENDING"],
        "active_flag": ["N", "Y"]
    }
}

NUMERIC_MINIMUMS = {
    "department_master": {"floor": 0, "capacity_beds": 1},
    "bed_master": {"floor": 0}
}

print("Healthcare validation rules are ready.")

## 5. Check each dataset and separate valid and rejected records

For every dataset, this step:
1. Keeps the original row as `_raw_record`.
2. Checks required fields, failed casts, categories, numeric ranges and timestamp order.
3. Marks invalid records as rejected.
4. Keeps valid records for the next step.

Rejected rows are saved for review; they are not silently deleted.

In [0]:
def prepare_dataset(name, df):
    # Keep the original Bronze row for audit and traceability.
    df = df.withColumn("_raw_record", F.to_json(F.struct(*[F.col(c) for c in df.columns])))

    reject_conditions = []
    reject_reasons = []
    rule_ids = []

    # Required fields
    for column in REQUIRED_FIELDS[name]:
        if column not in df.columns:
            raise ValueError(f"Missing required column: {name}.{column}")

        missing = F.col(column).isNull() | (F.trim(F.col(column).cast("string")) == "")
        reject_conditions.append(missing)
        reject_reasons.append(F.when(missing, F.lit("Missing required field: " + column)))
        rule_ids.append(F.when(missing, F.lit("DQ001")))

    # Cast failures: test the original value before replacing it with the converted value.
    for column, target_type in TYPE_MAP[name].items():
        if column in df.columns:
            raw_column = "__raw_" + column
            error_column = "__cast_error_" + column
            df = df.withColumn(raw_column, F.col(column))
            df = df.withColumn(column, F.col(column).cast(target_type))
            failed = F.col(raw_column).isNotNull() & F.col(column).isNull()
            reject_conditions.append(failed)
            reject_reasons.append(F.when(failed, F.lit("Invalid " + target_type + " value: " + column)))
            rule_ids.append(F.when(failed, F.lit("DQ002")))

    # Allowed category values
    for column, allowed in CATEGORY_RULES.get(name, {}).items():
        if column in df.columns:
            invalid = (F.col(column).isNotNull()
                       & (F.trim(F.col(column).cast("string")) != "")
                       & (~F.col(column).isin(allowed)))
            reject_conditions.append(invalid)
            reject_reasons.append(F.when(invalid, F.lit("Invalid category: " + column)))
            rule_ids.append(F.when(invalid, F.lit("DQ003")))

    # Numeric minimum values
    for column, minimum in NUMERIC_MINIMUMS.get(name, {}).items():
        invalid = F.col(column).isNotNull() & (F.col(column) < F.lit(minimum))
        reject_conditions.append(invalid)
        reject_reasons.append(F.when(invalid, F.lit("Invalid numeric range: " + column)))
        rule_ids.append(F.when(invalid, F.lit("DQ004")))

    # Timestamp order checks
    chronology = {
        "patient_encounters": [("admission_ts", "discharge_ts")],
        "lab_test_events": [("ordered_ts", "collected_ts"), ("collected_ts", "result_ts"), ("ordered_ts", "result_ts")],
        "bed_transfer_events": [("transfer_ts", "recorded_ts")]
    }

    for first, second in chronology.get(name, []):
        # Compare timestamps only when both columns are present in the source.
        if first not in df.columns or second not in df.columns:
            continue
        invalid = (F.col(first).isNotNull() & F.col(second).isNotNull()
                   & (F.col(second) < F.col(first)))
        reject_conditions.append(invalid)
        reject_reasons.append(F.when(invalid, F.lit(second + " is earlier than " + first)))
        rule_ids.append(F.when(invalid, F.lit("DQ005")))

    rejected_condition = reduce(lambda a, b: a | b, reject_conditions)
    df = df.withColumn("_reject_reason", F.concat_ws("; ", *reject_reasons))
    df = df.withColumn("_rule_ids", F.concat_ws(",", *rule_ids))
    df = df.withColumn("_is_rejected", rejected_condition)

    return df

prepared = {}
valid = {}
rejected_rows = []

for name, df in source_tables.items():
    result = prepare_dataset(name, df)
    prepared[name] = result
    valid[name] = result.filter(~F.col("_is_rejected"))

    rejected = result.filter(F.col("_is_rejected")).select(
        F.lit(name).alias("dataset"),
        F.col(BUSINESS_KEYS[name]).cast("string").alias("business_key"),
        F.col("_rule_ids").alias("rule_id"),
        F.col("_reject_reason").alias("reason"),
        F.col("source_file").cast("string").alias("source_file") if "source_file" in result.columns else F.lit(None).cast("string").alias("source_file"),
        F.col("ingestion_timestamp").cast("string").alias("ingested_at") if "ingestion_timestamp" in result.columns else F.lit(None).cast("string").alias("ingested_at"),
        F.col("_raw_record").alias("raw_record"),
        F.current_timestamp().alias("rejected_at")
    )
    rejected_rows.append(rejected)

    print(name, "| input:", df.count(), "| valid:", valid[name].count(), "| rejected:", rejected.count())

rejected_all = reduce(lambda a, b: a.unionByName(b), rejected_rows)

print("TOTAL REJECTED RECORDS:", rejected_all.count())
print("REJECTION REASONS:")
rejected_all.groupBy("reason").count().orderBy(F.desc("count")).show(30, truncate=False)

## 6. Remove duplicates and record what was removed

- Patient encounters: remove exact duplicate business records only.
- Other five datasets: keep one row per business key.
- Every removed row is written to the duplicate audit table.

This preserves the original healthcare notebook's deduplication approach.

In [0]:
DEDUP_ORDER = {
    "patient_encounters": [
        ("ingestion_timestamp", "desc"),
        ("source_file", "desc"),
    ],
    "lab_test_events": [
        ("ordered_ts", "desc"),
        ("result_ts", "desc"),
        ("ingestion_timestamp", "desc"),
        ("source_file", "desc"),
    ],
    "bed_transfer_events": [
        ("transfer_ts", "desc"),
        ("recorded_ts", "desc"),
        ("ingestion_timestamp", "desc"),
        ("source_file", "desc"),
    ],
    "bed_master": [
        ("effective_date", "desc"),
    ],
    "patient_master": [
        ("registration_date", "desc"),
    ],
}

EXACT_RECORD_DEDUP_DATASETS = ["patient_encounters"]
METADATA_COLUMNS = {"source_file", "ingestion_timestamp"}

def business_columns(df):
    # Business columns exclude ingestion metadata and notebook helper columns.
    return [
        c for c in df.columns
        if c not in METADATA_COLUMNS
        and not c.startswith("_")
        and not c.startswith("__")
    ]

def dedup_partition_columns(name, df):
    # Encounters: remove exact business-record copies only.
    if name in EXACT_RECORD_DEDUP_DATASETS:
        return business_columns(df)
    # Other datasets: keep one deterministic row per business key.
    return [BUSINESS_KEYS[name]]

def deduplicate(name, df):
    key = BUSINESS_KEYS[name]
    partition_columns = dedup_partition_columns(name, df)

    if name in DEDUP_ORDER:
        order_fields = [
            F.col(column_name).desc_nulls_last()
            if direction == "desc"
            else F.col(column_name).asc_nulls_last()
            for column_name, direction in DEDUP_ORDER[name]
            if column_name in df.columns
        ]
    else:
        order_fields = [
            F.col(column_name).desc_nulls_last()
            for column_name in sorted(df.columns)
            if column_name != key
            and column_name not in METADATA_COLUMNS
            and not column_name.startswith("_")
            and not column_name.startswith("__")
        ]

    # Add business columns as final tie-breakers for reproducible ordering.
    ordered_names = [
        column_name for column_name, direction in DEDUP_ORDER.get(name, [])
        if column_name in df.columns
    ]
    for column_name in sorted(business_columns(df)):
        if column_name not in ordered_names and column_name not in partition_columns:
            order_fields.append(F.col(column_name).desc_nulls_last())

    if not order_fields:
        order_fields = [F.col(key).asc_nulls_last()]

    window = Window.partitionBy(*partition_columns).orderBy(*order_fields)
    ranked = df.withColumn("_dedup_rank", F.row_number().over(window))

    duplicates = ranked.filter(F.col("_dedup_rank") > 1)
    kept = ranked.filter(F.col("_dedup_rank") == 1).drop("_dedup_rank")

    if name in EXACT_RECORD_DEDUP_DATASETS:
        rule_id = "EXACT_DUPLICATE_RECORD"
        reason = "Removed: all business columns identical to a kept row"
    else:
        rule_id = "DUPLICATE_BUSINESS_KEY"
        reason = "Removed by deterministic Silver deduplication"

    audit = duplicates.select(
        F.lit(name).alias("dataset"),
        F.col(key).cast("string").alias("business_key"),
        F.lit(rule_id).alias("rule_id"),
        F.lit(reason).alias("reason"),
        F.col("source_file").cast("string").alias("source_file")
            if "source_file" in duplicates.columns
            else F.lit(None).cast("string").alias("source_file"),
        F.col("_raw_record").alias("raw_record"),
        F.current_timestamp().alias("audited_at")
    )

    return kept, audit

silver_data = {}
duplicate_rows = []

for name, df in valid.items():
    clean, audit = deduplicate(name, df)
    silver_data[name] = clean
    duplicate_rows.append(audit)
    print(name, "| before deduplication:", df.count(),
          "| after deduplication:", clean.count(),
          "| duplicates removed:", audit.count())

duplicate_audit_all = reduce(lambda a, b: a.unionByName(b), duplicate_rows)

print("TOTAL DUPLICATES REMOVED:", duplicate_audit_all.count())
duplicate_audit_all.groupBy("dataset").count().orderBy("dataset").show()


## 7. Check relationships between healthcare tables

Check that:
- Encounters refer to existing patients, departments and beds.
- Lab events and transfers refer to existing encounters and patients.
- Transfer bed IDs and bed departments are valid.
- Event patient IDs match the patient on the parent encounter.
- Lab and transfer timestamps fall within the parent encounter dates.

These checks report mismatches for review. They do not silently delete records.

In [0]:
def require_columns(dataset_name, df, required_columns):
    missing = [column for column in required_columns if column not in df.columns]
    if missing:
        raise ValueError(
            f"Missing relationship/consistency columns in {dataset_name}: {missing}"
        )

def orphan_count(source_df, source_column, reference_df, reference_column):
    require_columns("source dataset", source_df, [source_column])
    require_columns("reference dataset", reference_df, [reference_column])
    source_keys = source_df.select(F.col(source_column).alias("source_key")).filter(F.col("source_key").isNotNull())
    reference_keys = reference_df.select(F.col(reference_column).alias("reference_key")).filter(F.col(reference_column).isNotNull()).distinct()
    return source_keys.join(reference_keys,
        source_keys.source_key == reference_keys.reference_key, "left_anti").count()

reference_checks = [
    ("patient_encounters", "patient_id", "patient_master", "patient_id"),
    ("patient_encounters", "department_id", "department_master", "department_id"),
    ("patient_encounters", "bed_id", "bed_master", "bed_id"),
    ("lab_test_events", "encounter_id", "patient_encounters", "encounter_id"),
    ("lab_test_events", "patient_id", "patient_master", "patient_id"),
    ("lab_test_events", "department_id", "department_master", "department_id"),
    ("bed_transfer_events", "encounter_id", "patient_encounters", "encounter_id"),
    ("bed_transfer_events", "patient_id", "patient_master", "patient_id"),
    ("bed_transfer_events", "department_id", "department_master", "department_id"),
    ("bed_transfer_events", "from_bed_id", "bed_master", "bed_id"),
    ("bed_transfer_events", "to_bed_id", "bed_master", "bed_id"),
    ("bed_master", "department_id", "department_master", "department_id")
]

# Fail early with a clear message if any configured relationship field is missing.
for source_name, source_col, ref_name, ref_col in reference_checks:
    require_columns(source_name, silver_data[source_name], [source_col])
    require_columns(ref_name, silver_data[ref_name], [ref_col])

reference_results = []
for source_name, source_col, ref_name, ref_col in reference_checks:
    missing = orphan_count(silver_data[source_name], source_col, silver_data[ref_name], ref_col)
    reference_results.append((source_name, source_col, ref_name, ref_col, missing))
    print(source_name + "." + source_col, "->", ref_name + "." + ref_col, "| unmatched:", missing)

reference_profile = spark.createDataFrame(reference_results,
    ["source_dataset", "source_column", "reference_dataset", "reference_column", "unmatched_rows"])
display(reference_profile)

# Preflight columns used by parent, department, bed and lifecycle checks.
consistency_columns = {
    "lab_test_events": ["encounter_id", "patient_id", "department_id", "ordered_ts"],
    "bed_transfer_events": ["encounter_id", "patient_id", "department_id", "to_bed_id", "transfer_ts"],
    "patient_encounters": ["encounter_id", "patient_id", "department_id", "bed_id", "admission_ts", "discharge_ts"],
    "bed_master": ["bed_id", "department_id"],
}
for dataset_name, columns in consistency_columns.items():
    require_columns(dataset_name, silver_data[dataset_name], columns)

# Check event patient IDs against the patient ID on the parent encounter.
for name in ["lab_test_events", "bed_transfer_events"]:
    event = silver_data[name].alias("event")
    encounter = silver_data["patient_encounters"].alias("encounter")
    mismatch = (event.join(encounter, F.col("event.encounter_id") == F.col("encounter.encounter_id"), "inner")
        .filter(F.col("event.patient_id") != F.col("encounter.patient_id")).count())
    print(name, "| patient does not match parent encounter:", mismatch)

# Check lab department against parent encounter department.
lab_department_mismatch = (silver_data["lab_test_events"].alias("event")
    .join(silver_data["patient_encounters"].alias("encounter"),
          F.col("event.encounter_id") == F.col("encounter.encounter_id"), "inner")
    .filter(F.col("event.department_id") != F.col("encounter.department_id")).count())
print("Lab department does not match encounter department:", lab_department_mismatch)

# Check transfer department against destination bed department.
transfer_department_mismatch = (silver_data["bed_transfer_events"].alias("transfer")
    .join(silver_data["bed_master"].alias("bed"),
          F.col("transfer.to_bed_id") == F.col("bed.bed_id"), "inner")
    .filter(F.col("transfer.department_id") != F.col("bed.department_id")).count())
print("Transfer department does not match destination bed department:", transfer_department_mismatch)

# Check encounter department against assigned bed department.
encounter_bed_mismatch = (silver_data["patient_encounters"].alias("encounter")
    .join(silver_data["bed_master"].alias("bed"),
          F.col("encounter.bed_id") == F.col("bed.bed_id"), "inner")
    .filter(F.col("encounter.department_id") != F.col("bed.department_id")).count())
print("Encounter department does not match assigned bed department:", encounter_bed_mismatch)

# Check event timestamps against the parent encounter lifecycle.
for name, event_column in [("lab_test_events", "ordered_ts"), ("bed_transfer_events", "transfer_ts")]:
    event = silver_data[name].alias("event")
    encounter = silver_data["patient_encounters"].alias("encounter")
    outside = (event.join(encounter, F.col("event.encounter_id") == F.col("encounter.encounter_id"), "inner")
        .filter((F.col("event." + event_column) < F.col("encounter.admission_ts")) |
                (F.col("encounter.discharge_ts").isNotNull() &
                 (F.col("event." + event_column) > F.col("encounter.discharge_ts")))).count())
    print(name, "| event outside encounter dates:", outside)

# Rule register: relationship mismatches are reported for review, not silently quarantined.
DQ_RULE_REGISTER = [
    ("DQ001", "All datasets", "Required fields", "Critical", "Quarantine"),
    ("DQ002", "All datasets", "Configured type conversion", "High", "Quarantine"),
    ("DQ003", "Configured categorical fields", "Allowed category values", "High", "Quarantine"),
    ("DQ004", "Numeric master fields", "Non-negative floor and positive bed capacity", "High", "Quarantine"),
    ("DQ005", "Encounter and event datasets", "Timestamp chronology", "High", "Quarantine"),
    ("DQ006", "All datasets", "Business-key duplicate handling", "High", "Audit and deduplicate"),
    ("DQ007", "Related datasets", "Foreign-key coverage", "High", "Report for review"),
    ("DQ008", "Encounter and event datasets", "Parent, department, bed and event-lifecycle consistency", "High", "Report for review"),
]
dq_rule_register_df = spark.createDataFrame(
    DQ_RULE_REGISTER,
    ["rule_id", "dataset_scope", "rule_description", "severity", "handling_action"]
)
display(dq_rule_register_df)


## 8. Write the six cleaned Silver tables

In [0]:
output_tables = {
    "silver_patient_encounters": silver_data["patient_encounters"],
    "silver_lab_test_events": silver_data["lab_test_events"],
    "silver_bed_transfer_events": silver_data["bed_transfer_events"],
    "silver_patients": silver_data["patient_master"],
    "silver_departments": silver_data["department_master"],
    "silver_beds": silver_data["bed_master"]
}

for table_name, df in output_tables.items():
    helper_columns = [c for c in df.columns
                      if c.startswith("__") or c in ["_reject_reason", "_rule_ids", "_is_rejected", "_dedup_rank"]]
    final_df = df.drop(*helper_columns)

    (final_df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver(table_name)))

    print("Created:", silver(table_name), "| rows:", final_df.count())

## 9. Write quarantine and duplicate audit tables

In [0]:
(rejected_all.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver("silver_rejected_records")))

(duplicate_audit_all.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver("silver_duplicate_audit")))

print("Created:", silver("silver_rejected_records"), "| rows:", rejected_all.count())
print("Created:", silver("silver_duplicate_audit"), "| rows:", duplicate_audit_all.count())

## 10. Final validation

This section checks:
- Each Silver table exists.
- Business keys are not NULL.
- Business keys are unique.
- Input rows = Silver rows + rejected rows + duplicate rows removed.
- Bronze tables were not changed by this notebook.

In [0]:
validation_results = []

for name, table_name in {
    "patient_encounters": "silver_patient_encounters",
    "lab_test_events": "silver_lab_test_events",
    "bed_transfer_events": "silver_bed_transfer_events",
    "patient_master": "silver_patients",
    "department_master": "silver_departments",
    "bed_master": "silver_beds"
}.items():
    full_name = silver(table_name)
    if not spark.catalog.tableExists(full_name):
        raise ValueError("Silver table was not created: " + full_name)

    df = spark.table(full_name)
    key = BUSINESS_KEYS[name]

    silver_count = df.count()
    null_keys = df.filter(F.col(key).isNull() | (F.trim(F.col(key).cast("string")) == "")).count()
    duplicate_keys = df.groupBy(key).count().filter(F.col("count") > 1).count()

    input_count = source_tables[name].count()
    rejected_count = rejected_all.filter(F.col("dataset") == name).count()
    duplicate_count = duplicate_audit_all.filter(F.col("dataset") == name).count()
    reconciled = silver_count + rejected_count + duplicate_count

    passed = (reconciled == input_count and null_keys == 0 and duplicate_keys == 0)
    validation_results.append((name, input_count, silver_count, rejected_count,
                               duplicate_count, null_keys, duplicate_keys, passed))

    print(name, "| input:", input_count, "| silver:", silver_count,
          "| rejected:", rejected_count, "| duplicates removed:", duplicate_count,
          "| null keys:", null_keys, "| duplicate keys:", duplicate_keys,
          "| reconciliation:", passed)

validation = spark.createDataFrame(validation_results, [
    "dataset", "input_rows", "silver_rows", "rejected_rows",
    "duplicates_removed", "null_business_keys", "duplicate_business_keys", "passed"
])
display(validation)

failed = validation.filter(~F.col("passed")).count()
if failed > 0:
    raise ValueError("Silver validation failed for " + str(failed) + " dataset(s).")

print("PASS: Silver row counts and business keys are validated.")

# Confirm this notebook did not change any Bronze Delta table.
bronze_versions_after = {
    name: get_table_version(bronze(table))
    for name, table in {
        "encounters": "bronze_patient_encounters",
        "labs": "bronze_lab_test_events",
        "transfers": "bronze_bed_transfer_events",
        "patients": "bronze_patients",
        "departments": "bronze_departments",
        "beds": "bronze_beds"
    }.items()
}

changed_bronze = [
    name for name in bronze_versions_before
    if bronze_versions_before[name] is not None
    and bronze_versions_after[name] != bronze_versions_before[name]
]

if changed_bronze:
    raise ValueError("Bronze tables were changed: " + str(changed_bronze))

print("PASS: Bronze tables remained unchanged where Delta history is available.")


## 11. Review the rejected and duplicate records

In [0]:
print("QUARANTINED RECORDS")
display(spark.table(silver("silver_rejected_records")).limit(50))

print("REMOVED DUPLICATE RECORDS")
display(spark.table(silver("silver_duplicate_audit")).limit(50))

## Silver layer – simple handoff summary

At the end of this notebook:

- Six cleaned healthcare tables are available in the Silver schema.
- Invalid records are stored in `silver_rejected_records`.
- Removed duplicate records are stored in `silver_duplicate_audit`.
- Reference mismatches are displayed for investigation and are not automatically deleted.
- Final row-count, NULL-key and duplicate-key checks are displayed.
- Bronze remains the read-only source.

**Not included:** A status CDC MERGE or clinician reference validation, because those source feeds/reference tables are not present in the supplied healthcare inputs.